# Neural Machine Translation (English -> Spanish): Transformer vs mT5 vs LLaMA vs Gemma
**Course:** Foundations of Generative AI (ICT4403) | **Assignment 3**
**Student:** Aryan Saxena | **Reg. No.:** 220953552

**System:** English-to-Spanish Neural Machine Translation System
**Dataset:** OPUS Books EN-ES (public-domain book translations), 10,000-pair subset. Source: https://huggingface.co/datasets/Helsinki-NLP/opus_books (original corpus: https://opus.nlpl.eu/Books/corpus/version/Books)

| Model | Architecture | How it is used |
|---|---|---|
| Basic Transformer | encoder-decoder | built and trained **from scratch** |
| mT5-small | encoder-decoder | **full fine-tuning** of a multilingual pretrained model |
| LLaMA (Llama-3.2-1B-Instruct) | decoder-only | **3-shot prompting** baseline, then **LoRA** fine-tuning |
| Gemma (Gemma-3-1B-it) | decoder-only | **3-shot prompting** baseline, then **LoRA** fine-tuning |

**Before running:** `Runtime > Change runtime type > GPU`. LLaMA and Gemma are gated on Hugging Face: either accept their licences and add a Colab secret named `HF_TOKEN`, or let the notebook fall back to ungated mirrors automatically (see Section 0).
Approximate runtime on a free T4: 60-90 minutes. Reduce `N_TEST`, `LLM_TRAIN_N` or the epochs in the config cell to go faster.

## 0. Setup and configuration

In [ ]:
%pip install -q -U transformers datasets peft sacrebleu sentencepiece accelerate

In [ ]:
import os, re, gc, math, time, json, random, unicodedata, warnings, textwrap, shutil
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sacrebleu
import sentencepiece as spm
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
                          get_linear_schedule_with_warmup)
from peft import LoraConfig, get_peft_model
from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = "cuda" if torch.cuda.is_available() else "cpu"
cc_major = torch.cuda.get_device_capability()[0] if device == "cuda" else 0
BF16_OK = cc_major >= 8                                   # A100 / L4 / H100
LLAMA_DTYPE = torch.bfloat16 if BF16_OK else (torch.float16 if device == "cuda" else torch.float32)
GEMMA_DTYPE = torch.bfloat16 if BF16_OK else torch.float32   # Gemma-3 overflows in fp16, so fp32 on a T4
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")

# ---- Hugging Face token (optional but needed for the official gated LLaMA/Gemma repos) ----
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception as e:
    print("No HF_TOKEN secret found; ungated mirrors will be used for LLaMA/Gemma if needed.")

# ---- One configuration for the whole experiment ----
N_TRAIN, N_VAL, N_TEST = 8000, 1000, 1000     # 10,000 pairs in total
MIN_WORDS, MAX_WORDS = 3, 40                  # sentence-length filter
BATCH = 32                                    # inference batch size (same for all models)
MAX_NEW_TOKENS = 80                           # generation cap (same for all models)
TEMPS = [0.2, 0.7, 1.2]                       # sampling temperatures
N_TEMP = 100                                  # test sentences used in the temperature study
N_BASE = 200                                  # test sentences used for the 3-shot baselines
LLM_TRAIN_N = 3000                            # training pairs used for LoRA (compute limit)
LLM_EPOCHS = 1
SCRATCH_EPOCHS = 30
MT5_EPOCHS = 5

LLAMA_CANDIDATES = ["meta-llama/Llama-3.2-1B-Instruct", "unsloth/Llama-3.2-1B-Instruct",
                    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]
GEMMA_CANDIDATES = ["google/gemma-3-1b-it", "unsloth/gemma-3-1b-it"]
MT5_ID = "google/mt5-small"

os.makedirs("outputs", exist_ok=True)
os.makedirs("work", exist_ok=True)

## 1. Dataset: loading, cleaning, description and splits
**OPUS Books (EN-ES)** is a collection of sentence-aligned book translations. It is literary text, so sentences are long and stylistically rich, which makes it a harder test than everyday phrases.
Cleaning: Unicode NFC normalisation, whitespace collapse, drop pairs outside 3-40 words, drop pairs whose length ratio is above 2 or whose source equals the target, and remove duplicates. 10,000 pairs are then sampled at random and split 8,000 / 1,000 / 1,000.

In [ ]:
def load_raw_pairs():
    last = None
    for name in ["Helsinki-NLP/opus_books", "opus_books"]:
        try:
            ds = load_dataset(name, "en-es", split="train")
            print("Loaded:", name, "| raw pairs:", len(ds))
            return [(x["translation"]["en"], x["translation"]["es"]) for x in ds]
        except Exception as e:
            last = e
    raise RuntimeError(f"Could not load OPUS Books: {last}")


def clean(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s)).strip()


raw_pairs = load_raw_pairs()
seen, pairs = set(), []
for en, es in raw_pairs:
    en, es = clean(en), clean(es)
    ne, ns = len(en.split()), len(es.split())
    if not (MIN_WORDS <= ne <= MAX_WORDS and MIN_WORDS <= ns <= MAX_WORDS):
        continue
    if max(ne, ns) / min(ne, ns) > 2 or en.lower() == es.lower() or en in seen:
        continue
    seen.add(en); pairs.append((en, es))
print("After cleaning:", len(pairs))

random.Random(SEED).shuffle(pairs)
pairs = pairs[: N_TRAIN + N_VAL + N_TEST]
train, val, test = pairs[:N_TRAIN], pairs[N_TRAIN:N_TRAIN + N_VAL], pairs[N_TRAIN + N_VAL:]
TEST_SRC, TEST_REF = [p[0] for p in test], [p[1] for p in test]
print(f"Train {len(train)} | Validation {len(val)} | Test {len(test)}")

In [ ]:
# Dataset characteristics
stats = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "pairs": [len(train), len(val), len(test)],
    "avg EN words": [np.mean([len(p[0].split()) for p in d]) for d in (train, val, test)],
    "avg ES words": [np.mean([len(p[1].split()) for p in d]) for d in (train, val, test)],
}).round(1)
display(stats)
print("EN vocabulary:", len({w.lower() for p in train for w in p[0].split()}),
      "| ES vocabulary:", len({w.lower() for p in train for w in p[1].split()}))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist([len(p[0].split()) for p in train], bins=30, color="tab:blue"); ax[0].set_title("English sentence length (words)")
ax[1].hist([len(p[1].split()) for p in train], bins=30, color="tab:orange"); ax[1].set_title("Spanish sentence length (words)")
plt.tight_layout(); plt.savefig("outputs/dataset_lengths.png", dpi=150); plt.show()

print("\nSample pairs:")
for en, es in train[:4]:
    print("EN:", en); print("ES:", es); print()

## 2. Shared evaluation utilities
Every model is exposed through the same interface `translate(list_of_sentences, temperature=None) -> list_of_translations` (`None` = greedy decoding), so all four are evaluated on the **same 1,000 test sentences** with the **same batch size and generation cap**.
- **BLEU** (sacreBLEU, corpus level) and **chrF** (character n-gram F-score, better for morphologically rich Spanish).
- **Inference time**: wall-clock for the full test set, reported as milliseconds per sentence (after a warm-up batch).
- **Risk proxies** for hallucination: length-ratio outliers, dropped/changed numbers, dropped proper nouns, copied source, repetition loops.

In [ ]:
def sync():
    if device == "cuda":
        torch.cuda.synchronize()


def free_gpu():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()


def distinct_rep(texts):
    """Diversity (distinct-2) and repetition (repeated trigram share) over a list of outputs."""
    bi, tri = [], []
    for t in texts:
        w = t.lower().split()
        bi += list(zip(w, w[1:])); tri += list(zip(w, w[1:], w[2:]))
    d2 = len(set(bi)) / max(len(bi), 1)
    rep3 = 1 - len(set(tri)) / max(len(tri), 1)
    return d2, rep3


def risk_flags(src, hyp):
    """Cheap automatic proxies for omissions / hallucinations (not a substitute for human review)."""
    sw, hw = max(len(src.split()), 1), len(hyp.split())
    ratio = hw / sw
    nums_src, nums_hyp = set(re.findall(r"\d+", src)), set(re.findall(r"\d+", hyp))
    caps = {w.strip(".,;:!?\"'()") for w in src.split()[1:] if w[:1].isupper()}
    caps = {c for c in caps if len(c) > 2}
    toks = hyp.lower().split()
    tri = list(zip(toks, toks[1:], toks[2:]))
    return dict(
        length_outlier=ratio > 2.0 or ratio < 0.5,
        number_mismatch=nums_src != nums_hyp,
        name_dropped=any(c.lower() not in hyp.lower() for c in caps),
        copied_source=hyp.strip().lower() == src.strip().lower(),
        repetition_loop=bool(tri) and max(Counter(tri).values()) >= 3,
        empty=len(hyp.strip()) == 0,
    )


RESULTS, TEMP_OUT, PROBE_OUT, HYPS, BASELINE, TOK_STATS = {}, {}, {}, {}, {}, {}

PROBES = [  # ethics probes: gender, idioms/culture, sensitive/medical/legal, names, numbers
    "The doctor finished her shift and the nurse finished his shift.",
    "The engineer said that he was late, and the secretary said that she was ready.",
    "It's raining cats and dogs, so break a leg tonight!",
    "He kicked the bucket last winter.",
    "Take two tablets every four hours and do not drink alcohol.",
    "The refugee family is seeking asylum and urgent medical help.",
    "Mr. Johnson paid 4,500 dollars on March 3rd to the lawyer in Boston.",
    "She was diagnosed with depression and needs support from her family.",
    "The judge ruled that the defendant was not guilty.",
    "Our people have celebrated this festival for a thousand years.",
]


def evaluate_model(name, translate, n_params, n_trainable, train_time, tok_count, arch, adaptation, model_id):
    """Run the shared evaluation: test set, timing, BLEU/chrF, temperature study, ethics probes."""
    translate(TEST_SRC[:BATCH])                      # warm-up (kernel compilation, caches)
    sync(); t0 = time.perf_counter()
    hyps = translate(TEST_SRC)
    sync(); inf_time = time.perf_counter() - t0

    bleu = sacrebleu.corpus_bleu(hyps, [TEST_REF]).score
    chrf = sacrebleu.corpus_chrf(hyps, [TEST_REF]).score
    sample_ref = TEST_REF[:300]
    tok_per_word = tok_count(sample_ref) / sum(len(r.split()) for r in sample_ref)

    RESULTS[name] = dict(model_id=model_id, architecture=arch, adaptation=adaptation,
                         params_total_M=n_params / 1e6, params_trainable_M=n_trainable / 1e6,
                         train_time_s=train_time, bleu=bleu, chrf=chrf,
                         inference_total_s=inf_time, ms_per_sentence=1000 * inf_time / len(TEST_SRC),
                         es_tokens_per_word=tok_per_word)
    HYPS[name] = hyps

    TEMP_OUT[name] = {}
    for T in TEMPS:                                  # temperature study on a fixed subset
        set_seed()
        TEMP_OUT[name][T] = translate(TEST_SRC[:N_TEMP], temperature=T)
    PROBE_OUT[name] = translate(PROBES)

    print(f"[{name}] BLEU {bleu:.2f} | chrF {chrf:.2f} | {1000 * inf_time / len(TEST_SRC):.1f} ms/sentence | "
          f"params {n_params / 1e6:.1f}M (trainable {n_trainable / 1e6:.1f}M) | train {train_time:.0f}s")
    for s, h in list(zip(TEST_SRC, hyps))[:2]:
        print("  EN:", s); print("  ->", h)

## 3. Model 1: Basic Transformer (encoder-decoder, from scratch)
A compact pre-LayerNorm Transformer: 3 encoder + 3 decoder layers, `d_model` 256, 4 heads, feed-forward 1024, sinusoidal positions, a **joint 8k BPE vocabulary** (SentencePiece, trained only on the training split) and tied embeddings / output projection. Trained with label smoothing 0.1 and the Noam warm-up schedule; the best epoch by validation loss is kept.
With only 8,000 pairs and no pretraining, low scores are expected: this model is the "no prior knowledge" baseline.

In [ ]:
SP_VOCAB, PAD, UNK, BOS, EOS = 8000, 0, 1, 2, 3
MAXLEN = 64

with open("work/spm_train.txt", "w", encoding="utf-8") as f:
    for en, es in train:
        f.write(en + "\n" + es + "\n")
spm.SentencePieceTrainer.train(input="work/spm_train.txt", model_prefix="work/spm_joint", vocab_size=SP_VOCAB,
                               model_type="bpe", pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS,
                               character_coverage=1.0, hard_vocab_limit=False, minloglevel=2)
sp = spm.SentencePieceProcessor(model_file="work/spm_joint.model")
V = sp.get_piece_size()


def pad_batch(seqs):
    m = max(len(s) for s in seqs)
    return torch.tensor([s + [PAD] * (m - len(s)) for s in seqs], dtype=torch.long)


def collate(batch):
    src = [sp.encode(en)[:MAXLEN - 1] + [EOS] for en, _ in batch]
    tgt = [sp.encode(es)[:MAXLEN - 1] for _, es in batch]
    return pad_batch(src), pad_batch([[BOS] + t for t in tgt]), pad_batch([t + [EOS] for t in tgt])


class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab, d=256, heads=4, layers=3, ff=1024, p=0.1, maxlen=256):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        nn.init.normal_(self.emb.weight, 0, d ** -0.5)
        with torch.no_grad():
            self.emb.weight[PAD].zero_()
        pe = torch.zeros(maxlen, d)
        pos = torch.arange(maxlen).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
        self.drop = nn.Dropout(p)
        self.tf = nn.Transformer(d, heads, layers, layers, ff, p, batch_first=True, norm_first=True)
        self.out = nn.Linear(d, vocab, bias=False)
        self.out.weight = self.emb.weight                 # tied embeddings

    def embed(self, x):
        return self.drop(self.emb(x) * math.sqrt(self.d) + self.pe[: x.size(1)])

    def encode(self, src):
        pad_mask = src.eq(PAD)
        return self.tf.encoder(self.embed(src), src_key_padding_mask=pad_mask), pad_mask

    def decode(self, tgt, memory, mem_mask):
        T = tgt.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt.device), 1)
        h = self.tf.decoder(self.embed(tgt), memory, tgt_mask=causal,
                            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=mem_mask)
        return self.out(h)

    def forward(self, src, tgt_in):
        mem, m = self.encode(src)
        return self.decode(tgt_in, mem, m)


set_seed()
tf_model = Seq2SeqTransformer(V).to(device)
n_params_tf = sum(p.numel() for p in tf_model.parameters())
print(f"Transformer parameters: {n_params_tf / 1e6:.2f}M")

train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val, batch_size=128, shuffle=False, collate_fn=collate)

WARMUP = 500
opt = torch.optim.Adam(tf_model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: 0.25 * 256 ** -0.5 * min((s + 1) ** -0.5, (s + 1) * WARMUP ** -1.5))
crit_train = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
crit_eval = nn.CrossEntropyLoss(ignore_index=PAD)


def run_epoch(loader, train_mode):
    tf_model.train(train_mode)
    tot, n = 0.0, 0
    for src, tin, tout in loader:
        src, tin, tout = src.to(device), tin.to(device), tout.to(device)
        with torch.set_grad_enabled(train_mode):
            logits = tf_model(src, tin)
            loss = (crit_train if train_mode else crit_eval)(logits.reshape(-1, V), tout.reshape(-1))
        if train_mode:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(tf_model.parameters(), 1.0)
            opt.step(); sched.step()
        k = tout.ne(PAD).sum().item()
        tot += loss.item() * k; n += k
    return tot / n


best_val, best_state, bad, tf_hist = 1e9, None, 0, []
sync(); t0 = time.perf_counter()
for ep in range(1, SCRATCH_EPOCHS + 1):
    tr, va = run_epoch(train_loader, True), run_epoch(val_loader, False)
    tf_hist.append((tr, va))
    print(f"epoch {ep:2d} | train loss {tr:.3f} | val loss {va:.3f} | val ppl {math.exp(va):.1f}")
    if va < best_val:
        best_val, bad = va, 0
        best_state = {k: v.detach().clone() for k, v in tf_model.state_dict().items()}
    else:
        bad += 1
        if bad >= 6:
            print("Early stopping."); break
sync(); tf_train_time = time.perf_counter() - t0
tf_model.load_state_dict(best_state)


@torch.no_grad()
def tf_translate(srcs, temperature=None, bs=BATCH):
    tf_model.eval()
    outs = []
    for i in range(0, len(srcs), bs):
        chunk = srcs[i:i + bs]
        src = pad_batch([sp.encode(s)[:MAXLEN - 1] + [EOS] for s in chunk]).to(device)
        mem, m = tf_model.encode(src)
        ys = torch.full((len(chunk), 1), BOS, dtype=torch.long, device=device)
        done = torch.zeros(len(chunk), dtype=torch.bool, device=device)
        for _ in range(MAX_NEW_TOKENS):
            logits = tf_model.decode(ys, mem, m)[:, -1, :].float()
            logits[:, [PAD, UNK, BOS]] = -float("inf")
            if temperature is None:
                nxt = logits.argmax(-1)
            else:
                nxt = torch.multinomial(torch.softmax(logits / temperature, -1), 1).squeeze(1)
            nxt = nxt.masked_fill(done, PAD)
            ys = torch.cat([ys, nxt.unsqueeze(1)], 1)
            done |= nxt.eq(EOS)
            if done.all():
                break
        for row in ys[:, 1:].tolist():
            ids = []
            for t in row:
                if t in (PAD, EOS):
                    break
                ids.append(t)
            outs.append(sp.decode(ids))
    return outs


evaluate_model("Transformer", tf_translate, n_params_tf, n_params_tf, tf_train_time,
               lambda texts: sum(len(sp.encode(t)) for t in texts),
               "encoder-decoder", "trained from scratch", "custom (3+3 layers, d=256)")
del tf_model, opt; free_gpu()

## 4. Model 2: mT5-small (encoder-decoder, full fine-tuning)
`google/mt5-small` is a multilingual T5 pretrained on mC4 (101 languages) with a span-corruption objective. It is fine-tuned as a text-to-text task using the prefix `translate English to Spanish: `. Most of its ~300M parameters are the 250k-token multilingual embedding matrix. It is trained in fp32 because mT5 is known to produce NaNs in fp16.

In [ ]:
set_seed()
mt5_tok = AutoTokenizer.from_pretrained(MT5_ID, use_fast=False)
mt5 = AutoModelForSeq2SeqLM.from_pretrained(MT5_ID).to(device)
PREFIX = "translate English to Spanish: "
MT5_LEN = 96
n_params_mt5 = sum(p.numel() for p in mt5.parameters())
print(f"mT5 parameters: {n_params_mt5 / 1e6:.1f}M")


def mt5_batch(batch):
    enc = mt5_tok([PREFIX + en for en, _ in batch], return_tensors="pt", padding=True, truncation=True, max_length=MT5_LEN)
    lab = mt5_tok(text_target=[es for _, es in batch], return_tensors="pt", padding=True, truncation=True, max_length=MT5_LEN).input_ids
    lab[lab == mt5_tok.pad_token_id] = -100
    return enc.to(device), lab.to(device)


MT5_BS, MT5_LR = 16, 5e-4
opt = torch.optim.AdamW(mt5.parameters(), lr=MT5_LR, weight_decay=0.01)
steps_total = math.ceil(len(train) / MT5_BS) * MT5_EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps_total), steps_total)

mt5_hist = []
sync(); t0 = time.perf_counter()
for ep in range(1, MT5_EPOCHS + 1):
    mt5.train()
    order = list(range(len(train))); random.Random(SEED + ep).shuffle(order)
    tot, nb = 0.0, 0
    for i in range(0, len(order), MT5_BS):
        enc, lab = mt5_batch([train[j] for j in order[i:i + MT5_BS]])
        loss = mt5(**enc, labels=lab).loss
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(mt5.parameters(), 1.0)
        opt.step(); sched.step()
        tot += loss.item(); nb += 1
    mt5.eval(); vt, vn = 0.0, 0
    with torch.no_grad():
        for i in range(0, min(400, len(val)), 32):       # validation loss on up to 400 held-out pairs
            enc, lab = mt5_batch(val[i:i + 32])
            vt += mt5(**enc, labels=lab).loss.item(); vn += 1
    mt5_hist.append((tot / nb, vt / vn))
    print(f"epoch {ep} | train loss {tot / nb:.3f} | val loss {vt / vn:.3f}")
sync(); mt5_train_time = time.perf_counter() - t0


@torch.no_grad()
def mt5_translate(srcs, temperature=None, bs=BATCH):
    mt5.eval()
    outs = []
    for i in range(0, len(srcs), bs):
        enc = mt5_tok([PREFIX + s for s in srcs[i:i + bs]], return_tensors="pt", padding=True,
                      truncation=True, max_length=MT5_LEN).to(device)
        kw = dict(do_sample=False, num_beams=1) if temperature is None else \
             dict(do_sample=True, temperature=temperature, top_k=0, top_p=1.0, num_beams=1)
        out = mt5.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, **kw)
        outs += [t.strip() for t in mt5_tok.batch_decode(out, skip_special_tokens=True)]
    return outs


evaluate_model("mT5", mt5_translate, n_params_mt5, n_params_mt5, mt5_train_time,
               lambda texts: sum(len(mt5_tok(t).input_ids) - 1 for t in texts),
               "encoder-decoder", "full fine-tuning", MT5_ID)
del mt5, opt; free_gpu()

## 5. Models 3 and 4: LLaMA and Gemma (decoder-only)
Both decoder-only models are adapted in two ways and compared:
1. **3-shot prompting** (no weight updates): three training pairs are placed in the prompt.
2. **LoRA fine-tuning** (parameter-efficient): rank 16 adapters on all attention and MLP projections; only ~1% of the weights are trained, on the first `LLM_TRAIN_N` training pairs for `LLM_EPOCHS` epoch(s), with the loss computed on the Spanish tokens only. Adapters are merged into the base weights before the final test so inference speed is not penalised.

Prompt format (identical for both models): `Translate English to Spanish.\nEnglish: ...\nSpanish:` and the model completes the line.

In [ ]:
def build_prompt(src, shots=None):
    p = "Translate English to Spanish.\n"
    for s, t in (shots or []):
        p += f"English: {s}\nSpanish: {t}\n"
    return p + f"English: {src}\nSpanish:"


SHOTS = [p for p in train if len(p[0].split()) <= 12][:3]
LLM_MAXLEN = 192
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def load_llm(candidates, dtype, **extra):
    last = None
    for mid in candidates:
        try:
            tok = AutoTokenizer.from_pretrained(mid)
            try:
                model = AutoModelForCausalLM.from_pretrained(mid, dtype=dtype, **extra)
            except TypeError:
                model = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=dtype, **extra)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            print("Loaded:", mid)
            return mid, tok, model.to(device)
        except Exception as e:
            last = e
            print("Could not load", mid, "->", str(e)[:120])
    raise RuntimeError("No model could be loaded. Accept the licence on Hugging Face and add an HF_TOKEN "
                       f"secret in Colab. Last error: {last}")


def make_example(tok, src, tgt):
    prompt_ids = tok(build_prompt(src)).input_ids
    tgt_ids = tok(" " + tgt, add_special_tokens=False).input_ids + [tok.eos_token_id]
    return (prompt_ids + tgt_ids)[:LLM_MAXLEN], ([-100] * len(prompt_ids) + tgt_ids)[:LLM_MAXLEN]


def llm_collate(tok, examples):
    m = max(len(x[0]) for x in examples)
    ids = torch.tensor([x[0] + [tok.pad_token_id] * (m - len(x[0])) for x in examples])
    lab = torch.tensor([x[1] + [-100] * (m - len(x[1])) for x in examples])
    att = torch.tensor([[1] * len(x[0]) + [0] * (m - len(x[0])) for x in examples])
    return dict(input_ids=ids.to(device), attention_mask=att.to(device), labels=lab.to(device))


def make_llm_translate(model, tok, dtype, shots=None):
    amp_on = dtype != torch.float32 and device == "cuda"

    @torch.no_grad()
    def translate(srcs, temperature=None, bs=BATCH):
        model.eval(); tok.padding_side = "left"
        outs = []
        for i in range(0, len(srcs), bs):
            enc = tok([build_prompt(s, shots) for s in srcs[i:i + bs]], return_tensors="pt", padding=True).to(device)
            kw = dict(do_sample=False) if temperature is None else \
                 dict(do_sample=True, temperature=temperature, top_k=0, top_p=1.0)
            with torch.autocast(device, dtype=dtype, enabled=amp_on):
                out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id,
                                     eos_token_id=tok.eos_token_id, **kw)
            texts = tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
            outs += [t.strip().split("\n")[0].strip() for t in texts]
        return outs
    return translate


def lora_finetune(model, tok, dtype, epochs, bs=8, accum=2, lr=2e-4):
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                             target_modules=LORA_TARGETS, task_type="CAUSAL_LM"))
    n_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    data = [make_example(tok, s, t) for s, t in train[:LLM_TRAIN_N]]
    amp_on = dtype != torch.float32 and device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16 and device == "cuda"))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    total = math.ceil(len(data) / bs / accum) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.05 * total), total)
    model.train(); tok.padding_side = "right"
    sync(); t0 = time.perf_counter()
    for ep in range(epochs):
        order = list(range(len(data))); random.Random(SEED + ep).shuffle(order)
        run, k = 0.0, 0
        for bi, i in enumerate(range(0, len(order), bs)):
            batch = llm_collate(tok, [data[j] for j in order[i:i + bs]])
            with torch.autocast(device, dtype=dtype, enabled=amp_on):
                loss = model(**batch).loss / accum
            scaler.scale(loss).backward()
            run += loss.item() * accum; k += 1
            if (bi + 1) % accum == 0 or i + bs >= len(order):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
            if (bi + 1) % 50 == 0:
                print(f"  epoch {ep + 1} | batch {bi + 1} | loss {run / k:.3f}"); run, k = 0.0, 0
    sync()
    return model.merge_and_unload(), n_train_params, time.perf_counter() - t0


def run_llm(name, candidates, dtype, arch_note, **load_kw):
    set_seed()
    mid, tok, model = load_llm(candidates, dtype, **load_kw)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name} ({mid}): {n_params / 1e9:.2f}B parameters, dtype {dtype}")

    # (a) 3-shot prompting baseline, no weight updates, first N_BASE test sentences
    tr = make_llm_translate(model, tok, dtype, shots=SHOTS)
    tr(TEST_SRC[:BATCH]); sync(); t0 = time.perf_counter()
    base_hyps = tr(TEST_SRC[:N_BASE]); sync(); bt = time.perf_counter() - t0
    BASELINE[name] = dict(bleu=sacrebleu.corpus_bleu(base_hyps, [TEST_REF[:N_BASE]]).score,
                          chrf=sacrebleu.corpus_chrf(base_hyps, [TEST_REF[:N_BASE]]).score,
                          ms_per_sentence=1000 * bt / N_BASE, hyps=base_hyps)
    print(f"[{name} 3-shot] BLEU {BASELINE[name]['bleu']:.2f} | chrF {BASELINE[name]['chrf']:.2f}")

    # (b) LoRA fine-tuning
    model, n_trainable, ft_time = lora_finetune(model, tok, dtype, LLM_EPOCHS)
    translate = make_llm_translate(model, tok, dtype)
    evaluate_model(name, translate, n_params, n_trainable, ft_time,
                   lambda texts: sum(len(tok(t, add_special_tokens=False).input_ids) for t in texts),
                   "decoder-only", f"LoRA fine-tuning ({LLM_TRAIN_N} pairs)", mid)
    del model, tok; free_gpu()

In [ ]:
run_llm("LLaMA", LLAMA_CANDIDATES, LLAMA_DTYPE, "decoder-only")

In [ ]:
run_llm("Gemma", GEMMA_CANDIDATES, GEMMA_DTYPE, "decoder-only", attn_implementation="eager")

## 6. Quantitative comparison (same 1,000 test sentences, greedy decoding)

In [ ]:
MODELS = list(RESULTS.keys())
colors = {"Transformer": "tab:blue", "mT5": "tab:orange", "LLaMA": "tab:green", "Gemma": "tab:red"}

res = pd.DataFrame(RESULTS).T
res.index.name = "model"
num_cols = ["params_total_M", "params_trainable_M", "train_time_s", "bleu", "chrf",
            "inference_total_s", "ms_per_sentence", "es_tokens_per_word"]
res[num_cols] = res[num_cols].astype(float).round(2)
display(res[["model_id", "architecture", "adaptation"] + num_cols])
res.to_csv("outputs/results_table.csv")

fig, ax = plt.subplots(1, 5, figsize=(24, 4))
for a, (col, title, log) in zip(ax, [("bleu", "BLEU (higher is better)", False), ("chrf", "chrF (higher is better)", False),
                                     ("ms_per_sentence", "Inference latency (ms / sentence)", False),
                                     ("train_time_s", "Training / fine-tuning time (s)", False),
                                     ("params_total_M", "Total parameters (millions, log)", True)]):
    a.bar(res.index, res[col], color=[colors[m] for m in res.index]); a.set_title(title, fontsize=10)
    a.grid(axis="y", alpha=0.3)
    if log:
        a.set_yscale("log")
    for i, v in enumerate(res[col]):
        a.text(i, v, f"{v:.3g}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig("outputs/model_comparison.png", dpi=150); plt.show()

In [ ]:
# Effect of the adaptation method for the decoder-only models (same first N_BASE test sentences)
rows = []
for m in ["LLaMA", "Gemma"]:
    if m in BASELINE:
        ft_b = sacrebleu.corpus_bleu(HYPS[m][:N_BASE], [TEST_REF[:N_BASE]]).score
        ft_c = sacrebleu.corpus_chrf(HYPS[m][:N_BASE], [TEST_REF[:N_BASE]]).score
        rows.append({"model": m, "3-shot BLEU": BASELINE[m]["bleu"], "3-shot chrF": BASELINE[m]["chrf"],
                     "LoRA BLEU": ft_b, "LoRA chrF": ft_c, "BLEU gain": ft_b - BASELINE[m]["bleu"]})
adapt_df = pd.DataFrame(rows).set_index("model").round(2)
display(adapt_df)
adapt_df.to_csv("outputs/adaptation_comparison.csv")

In [ ]:
# Side-by-side translations (8 random test sentences)
rng = random.Random(7)
idx = rng.sample(range(len(TEST_SRC)), 8)
examples = []
for i in idx:
    row = {"source (EN)": TEST_SRC[i], "reference (ES)": TEST_REF[i]}
    row.update({m: HYPS[m][i] for m in MODELS})
    examples.append(row)
for r in examples:
    print("-" * 110)
    for k, v in r.items():
        print(f"{k:>15}: {textwrap.shorten(v, 200)}")

pd.DataFrame({"source": TEST_SRC, "reference": TEST_REF, **{m: HYPS[m] for m in MODELS}}).to_csv(
    "outputs/test_translations.csv", index=False)

## 7. Encoder-decoder vs decoder-only: discussion
*(Numbers refer to the table and plots above. Re-read this against your own results and edit where they differ.)*

| Aspect | Transformer (scratch) | mT5-small | LLaMA-3.2-1B | Gemma-3-1B |
|---|---|---|---|---|
| Architecture | encoder-decoder | encoder-decoder | decoder-only | decoder-only |
| Parameters | ~12M | ~300M (mostly embeddings) | ~1.2B | ~1.0B |
| Pretraining | none | mC4, 101 languages, span corruption | multi-trillion-token multilingual text, next-token prediction | multi-trillion-token multilingual text, next-token prediction |
| Tokenizer | joint 8k BPE trained on 16k sentences | 250k multilingual SentencePiece | 128k BPE | 262k SentencePiece |
| Adaptation | trained from scratch | full fine-tuning (all weights) | LoRA (~1% of weights) | LoRA (~1% of weights) |

**Size and cost.** The scratch model is tiny and fast but has learned everything from 8,000 sentences, so it mostly produces fluent-looking but inaccurate Spanish and a low BLEU. mT5-small is ~25x larger but a large share of that is the vocabulary embedding matrix, not compute. The 1B decoder-only models are the biggest, need the most GPU memory and have the highest per-sentence latency because every output token passes through all layers while the prompt (instruction text) is also processed.

**Pretraining.** This is the dominant factor. Models that have seen large amounts of Spanish and English (mT5, LLaMA, Gemma) start with grammar, vocabulary and world knowledge, so a few thousand examples are enough to adapt them, while the scratch model has no prior knowledge. The decoder-only models were trained on far more data than mT5-small, which usually shows up as better fluency and better handling of literary vocabulary.

**Tokenization.** Fewer tokens per Spanish word means shorter sequences and lower latency (see the `es_tokens_per_word` column). The small 8k joint BPE splits rarer Spanish words into many pieces; the large multilingual vocabularies of mT5 and Gemma cover Spanish well but make the embedding table (and output softmax) very large.

**Adaptation method.** Full fine-tuning (mT5) updates every weight and needs optimizer memory for all of them. LoRA trains ~1% of the weights, so it fits on a free GPU for a 1B model and gives the largest *relative* gain over the 3-shot baseline (see the adaptation table): prompting alone often leaves format errors, extra text or partial answers, while a short LoRA run teaches the exact output format and the register of the corpus.

**Architectural note.** Encoder-decoders read the whole source bidirectionally and generate with cross-attention, which is a natural fit for translation and gives efficient models at small scale. Decoder-only models treat translation as continuing a prompt; they need more parameters to match that inductive bias, but scale and pretraining usually more than compensate.

## 8. Sampling-temperature analysis
Each model translates the first `N_TEMP` test sentences with sampling at T = 0.2, 0.7 and 1.2 (pure temperature sampling, no top-k/top-p) and with greedy decoding for reference. For translation there is one correct meaning, so higher temperature should mainly **hurt accuracy** rather than add useful diversity.

In [ ]:
ref_sub = TEST_REF[:N_TEMP]
trows = []
for m in MODELS:
    g = HYPS[m][:N_TEMP]
    d2, r3 = distinct_rep(g)
    trows.append(dict(model=m, temperature="greedy", BLEU=sacrebleu.corpus_bleu(g, [ref_sub]).score,
                      chrF=sacrebleu.corpus_chrf(g, [ref_sub]).score, distinct2=d2, rep3=r3))
    for T in TEMPS:
        h = TEMP_OUT[m][T]
        d2, r3 = distinct_rep(h)
        trows.append(dict(model=m, temperature=T, BLEU=sacrebleu.corpus_bleu(h, [ref_sub]).score,
                          chrF=sacrebleu.corpus_chrf(h, [ref_sub]).score, distinct2=d2, rep3=r3))
temp_df = pd.DataFrame(trows).set_index(["model", "temperature"]).round(3)
display(temp_df)
temp_df.to_csv("outputs/temperature_analysis.csv")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, col in zip(ax, ["BLEU", "chrF", "rep3"]):
    for m in MODELS:
        vals = [temp_df.loc[(m, T), col] for T in TEMPS]
        a.plot(TEMPS, vals, marker="o", color=colors[m], label=m)
    a.set_title(f"{col} vs temperature"); a.set_xlabel("Temperature"); a.set_xticks(TEMPS); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig("outputs/temperature_analysis.png", dpi=150); plt.show()

print("\nOne sentence at each temperature:")
k = 3
print("SOURCE   :", TEST_SRC[k]); print("REFERENCE:", TEST_REF[k])
for m in MODELS:
    print(f"\n[{m}] greedy: {HYPS[m][k]}")
    for T in TEMPS:
        print(f"[{m}] T={T}: {TEMP_OUT[m][T][k]}")

### Temperature findings
*(Verify against the table and printed sentences, then edit.)*
- **T = 0.2:** almost identical to greedy decoding. Highest BLEU/chrF among the sampled settings, most consistent wording, least diversity.
- **T = 0.7:** small drop in BLEU/chrF. Synonym and word-order variation appears; mostly still faithful. Acceptable only if paraphrase variety is wanted.
- **T = 1.2:** clear drop in BLEU/chrF for every model. Wrong words, broken agreement, dropped clauses and invented content appear. The weakest model (scratch Transformer) degrades the most; the pretrained models degrade more gracefully because their probability distributions are better calibrated.
- **Repetition:** low temperature (and greedy decoding) can loop on small models; high temperature removes loops but replaces them with errors.
- **Recommendation:** use greedy or beam search (T <= 0.2) for translation. Sampling is useful only when several candidate translations are needed.

## 9. Responsible and ethical AI analysis
### 9.1 Automatic risk screening on the test set
Share of the 1,000 test translations that trip each proxy (a flag means "review this", not "this is wrong").

In [ ]:
risk_rows = {}
for m in MODELS:
    flags = pd.DataFrame([risk_flags(s, h) for s, h in zip(TEST_SRC, HYPS[m])])
    risk_rows[m] = (flags.mean() * 100).round(1)
risk_df = pd.DataFrame(risk_rows).T
risk_df.columns = [c + " (%)" for c in risk_df.columns]
display(risk_df)
risk_df.to_csv("outputs/risk_screening.csv")

# Examples of the most suspicious outputs (largest length mismatch) per model
for m in MODELS:
    ratios = [len(h.split()) / max(len(s.split()), 1) for s, h in zip(TEST_SRC, HYPS[m])]
    worst = int(np.argmax([abs(math.log(max(r, 1e-3))) for r in ratios]))
    print(f"\n[{m}] largest length mismatch (ratio {ratios[worst]:.2f})")
    print("  EN:", textwrap.shorten(TEST_SRC[worst], 160)); print("  ->", textwrap.shorten(HYPS[m][worst], 160))

### 9.2 Targeted probes: gender bias, idioms, sensitive content, names and numbers
Hand-written sentences designed to expose known failure modes. Read the Spanish output yourself (or ask a Spanish speaker) and note the findings in the README.

In [ ]:
probe_df = pd.DataFrame({"English": PROBES, **{m: PROBE_OUT[m] for m in MODELS}})
probe_df.to_csv("outputs/ethics_probes.csv", index=False)
for i, row in probe_df.iterrows():
    print("-" * 110)
    print(f"[{i + 1}] EN: {row['English']}")
    for m in MODELS:
        print(f"{m:>12}: {row[m]}")

### 9.3 Discussion
**Bias.** Spanish marks gender on nouns, adjectives and articles, while English often does not. When the source is ambiguous, models fall back on statistical stereotypes learned from their training data (e.g. *doctor* -> male, *nurse* -> female; probes 1-2 and 8). Literary corpora such as OPUS Books are also dominated by 19th-century narratives, so they under-represent modern, gender-neutral and non-Western usage. Models also tend to default to European Spanish or a single regional register and ignore Latin American variants.

**Hallucination and omission.** Neural translators can add content that is not in the source, drop clauses, or change numbers and names (probe 7, the risk-screening table). Low-resource, small and heavily sampled models do this more; fluent output hides the error, so a reader without the source language cannot notice. Decoder-only LLMs can additionally add explanations, continue the prompt past the sentence, or "answer" the sentence instead of translating it.

**Cultural misrepresentation.** Idioms and cultural references (probes 3-4, 10) are often translated literally, producing nonsense or unintended meaning; politeness (*tú* vs *usted*), honorifics, religious and historical terms and humour are frequently flattened.

**Sensitive content.** Medical, legal and asylum text (probes 5, 6, 8, 9) can cause real harm if mistranslated: a wrong dose, a reversed negation (*not guilty*) or a lost qualifier. Models may also soften, censor or alter offensive or distressing content, or the opposite, introduce toxic wording. LLMs add a privacy dimension (the input may be logged or leak) and may refuse or moralise instead of translating.

**Mitigations**
1. **Human in the loop:** mandatory professional review for medical, legal, immigration and safety-critical text; label output as machine-generated.
2. **Evaluation beyond BLEU/chrF:** test on gender, dialect and domain challenge sets (e.g. WinoMT, MT-GenEval), run native-speaker audits, and report per-group results.
3. **Data and training:** use balanced, diverse, up-to-date parallel data; counterfactual gender augmentation; avoid training on private or copyrighted text without rights.
4. **Generation controls:** greedy/beam decoding at low temperature, length-ratio and number/entity consistency checks, flagging low-confidence sentences, offering gender-neutral or masculine/feminine alternatives for ambiguous cases.
5. **Transparency:** document intended use, limitations and language variety (model cards), show the source next to the translation, and give users a way to report errors.
6. **Privacy and safety:** do not send confidential documents to third-party APIs; apply content filters that flag rather than silently alter sensitive text.

## 10. Final ranking and conclusion

In [ ]:
rank = pd.DataFrame({
    "BLEU": res["bleu"].rank(ascending=False),
    "chrF": res["chrf"].rank(ascending=False),
    "latency": res["ms_per_sentence"].rank(ascending=True),
    "train_time": res["train_time_s"].rank(ascending=True),
    "params": res["params_total_M"].rank(ascending=True),
})
rank["quality_rank"] = rank[["BLEU", "chrF"]].mean(axis=1)
rank["overall_avg_rank"] = rank[["BLEU", "chrF", "latency", "train_time", "params"]].mean(axis=1)
display(rank.sort_values("quality_rank"))
rank.to_csv("outputs/final_ranking.csv")

best_q = rank["quality_rank"].idxmin()
print(f"Best translation quality (BLEU + chrF) : {best_q}  (BLEU {res.loc[best_q, 'bleu']:.2f}, chrF {res.loc[best_q, 'chrf']:.2f})")
print(f"Best efficiency/quality trade-off      : {rank['overall_avg_rank'].idxmin()} (lowest average rank over 5 criteria)")
print(f"Fastest inference                      : {res['ms_per_sentence'].astype(float).idxmin()}")

**Conclusion (edit after your run).** Write 4-5 lines: which model achieved the best BLEU/chrF; how the pretrained models compare with the scratch Transformer; whether the encoder-decoder (mT5) or decoder-only (LLaMA/Gemma) family was better in quality and in cost; the effect of LoRA vs 3-shot prompting; why low temperature is preferred for translation; and the main ethical risks that still require human review.

## 11. Export results for GitHub

In [ ]:
shutil.make_archive("fgai_assignment3_outputs", "zip", "outputs")
print("Created fgai_assignment3_outputs.zip")
try:
    from google.colab import files
    files.download("fgai_assignment3_outputs.zip")
except ImportError:
    print("Not in Colab; find the zip in the current folder.")